In [1]:
from datasets import load_dataset
import pandas as pd

TARGET_ROWS = 250000
print(f"Streaming up to {TARGET_ROWS:,} clean labeled rows from Hugging Face...")

streamed_data = load_dataset("jonaskoenig/topic_classification", split="train", streaming=True)

clean_rows = []
topic_cols = None

for row in streamed_data:
    if topic_cols is None:
        topic_cols = [k for k in row.keys() if k != 'text']
    
    # Keep only rows that have valid text and at least 1 active topic tag
    text = row.get('text', '')
    if isinstance(text, str) and len(text.strip()) > 20:
        if sum(row[col] for col in topic_cols) > 0:
            clean_rows.append(row)
            
    if len(clean_rows) % 50000 == 0 and len(clean_rows) > 0:
        print(f"Collected {len(clean_rows):,} clean rows...")
        
    if len(clean_rows) >= TARGET_ROWS:
        break

df = pd.DataFrame(clean_rows)
df.to_csv("topic_dataset.csv", index=False)
print(f"SUCCESS! Saved 'topic_dataset.csv' with {len(df):,} clean rows and {len(topic_cols)} topics.")

e:\Machine_Learning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Streaming up to 250,000 clean labeled rows from Hugging Face...


Collected 50,000 clean rows...
Collected 100,000 clean rows...
Collected 150,000 clean rows...
Collected 200,000 clean rows...
Collected 250,000 clean rows...
SUCCESS! Saved 'topic_dataset.csv' with 250,000 clean rows and 23 topics.


In [3]:
from datasets import load_dataset
import pandas as pd

print("1. Loading existing 250k multi-label dataset...")
base_df = pd.read_csv("topic_dataset.csv")
topic_cols = [col for col in base_df.columns if col != 'text']
print(f"Base rows: {len(base_df):,} | Columns: {len(topic_cols)}")

new_rows = []

# --- SOURCE 2: Global News (AG News with full namespace) ---
print("2. Streaming 60,000 real global news articles (fancyzhx/ag_news)...")
ag_data = load_dataset("fancyzhx/ag_news", split="train", streaming=True)
ag_map = {
    0: ["politics", "news"],          # World -> politics + news
    1: ["sports", "news"],            # Sports -> sports + news
    2: ["business", "economy"],       # Business -> business + economy
    3: ["technology", "science"]      # Sci/Tech -> technology + science
}

for i, item in enumerate(ag_data):
    if i >= 60000:
        break
    row_dict = {col: 0 for col in topic_cols}
    row_dict["text"] = item["text"]
    for t in ag_map[item["label"]]:
        if t in row_dict:
            row_dict[t] = 1
    new_rows.append(row_dict)

# --- SOURCE 3: Indian News (IINOVAII/news-classification) ---
print("3. Streaming Indian news dataset (IINOVAII/news-classification)...")
in_data = load_dataset("IINOVAII/news-classification", split="train", streaming=True)
in_map = {
    "business": ["business", "economy", "news"],
    "technology": ["technology", "phones", "news"],
    "sports": ["sports", "news"],
    "politics": ["politics", "news"],
    "entertainment": ["movies", "music", "news"],
    "science": ["science", "news"],
    "health": ["health", "news"]
}

for item in in_data:
    cat = str(item.get("output", "")).strip().lower()
    headline = str(item.get("headlines", ""))
    desc = str(item.get("description", ""))
    full_text = f"{headline}. {desc}".strip()
    
    if len(full_text) > 20 and cat in in_map:
        row_dict = {col: 0 for col in topic_cols}
        row_dict["text"] = full_text
        for t in in_map[cat]:
            if t in row_dict:
                row_dict[t] = 1
        new_rows.append(row_dict)

news_df = pd.DataFrame(new_rows)
print(f"Collected {len(news_df):,} real global + Indian news articles!")

# Combine and shuffle both datasets
merged_df = pd.concat([base_df, news_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
merged_df.to_csv("topic_dataset.csv", index=False)
print(f"SUCCESS! Unified dataset saved with {len(merged_df):,} total rows across {len(topic_cols)} topics.")

1. Loading existing 250k multi-label dataset...
Base rows: 250,000 | Columns: 23
2. Streaming 60,000 real global news articles (fancyzhx/ag_news)...


e:\Machine_Learning\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub\datasets--fancyzhx--ag_news. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting GET https://huggingfa

3. Streaming Indian news dataset (IINOVAII/news-classification)...


e:\Machine_Learning\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub\datasets--IINOVAII--news-classification. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Collected 68,000 real global + Indian news articles!
SUCCESS! Unified dataset saved with 318,000 total rows across 23 topics.
